# Day 5 Exam: Architecture & Video Generation

**Coverage:** Notebooks 10-13 (DiT, guidance/control, cascaded diffusion, video generation), with review of previous days.

---

## Exam Rules

- **Total time: ~2 hours** (warmups ~15 min, main problems ~90 min)
- **No documentation, no Google, no LLM assistance.** Closed-book.
- **Honor system.** If you can't remember something, leave a comment and move on.
- Write clean, typed, documented code — as if submitting for code review.
- Run all test/assertion cells to validate your work before finishing.
- If stuck on a warmup for >7 min, move on. Main problems matter more.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import math

---

## Warmup Section (~15 minutes total)

Quick recall problems. 5 minutes each. Don't overthink these.

### Warmup 1: Adaptive Layer Norm (5 min)

Implement `adaptive_layer_norm(x, scale, shift)` as a standalone function.

**Behavior:**
1. Apply standard `LayerNorm` to `x` (normalize over the last dimension D)
2. Modulate the result: `output = normalized * scale + shift`

**Shapes:**
- `x`: `(B, T, D)` — input features
- `scale`: `(B, D)` — per-sample, per-feature scale (needs unsqueeze to broadcast over T)
- `shift`: `(B, D)` — per-sample, per-feature shift
- Output: `(B, T, D)`

**Notes:**
- Use `F.layer_norm` with `normalized_shape=[D]` for the base normalization
- Scale and shift are external conditioning vectors, not learned affine params

In [ ]:
# Warmup 1: Your implementation



In [ ]:
# Warmup 1: Tests
torch.manual_seed(42)
B, T, D = 2, 8, 64
x = torch.randn(B, T, D)
scale = torch.randn(B, D)
shift = torch.randn(B, D)

out = adaptive_layer_norm(x, scale, shift)
assert out.shape == (B, T, D), f"Expected shape {(B, T, D)}, got {out.shape}"

# scale=1, shift=0 should give standard LayerNorm
ones_scale = torch.ones(B, D)
zeros_shift = torch.zeros(B, D)
out_identity = adaptive_layer_norm(x, ones_scale, zeros_shift)
expected = F.layer_norm(x, [D])
assert torch.allclose(out_identity, expected, atol=1e-5), "scale=1, shift=0 should match standard LayerNorm"

print("Warmup 1 passed.")

### Warmup 2: Classifier-Free Guidance (5 min)

Implement `apply_cfg(uncond, cond, guidance_scale)` — the CFG formula.

**Formula:** `output = uncond + guidance_scale * (cond - uncond)`

**Shapes:**
- `uncond`: any tensor shape (unconditional model output)
- `cond`: same shape (conditional model output)
- `guidance_scale`: float scalar
- Output: same shape

In [ ]:
# Warmup 2: Your implementation



In [ ]:
# Warmup 2: Tests
torch.manual_seed(42)
uncond = torch.randn(4, 3, 32, 32)
cond = torch.randn(4, 3, 32, 32)

# guidance_scale=1 should give cond
out_s1 = apply_cfg(uncond, cond, guidance_scale=1.0)
assert torch.allclose(out_s1, cond), "guidance_scale=1 should return cond"

# guidance_scale=0 should give uncond
out_s0 = apply_cfg(uncond, cond, guidance_scale=0.0)
assert torch.allclose(out_s0, uncond), "guidance_scale=0 should return uncond"

# Arbitrary scale should follow the formula
out_s75 = apply_cfg(uncond, cond, guidance_scale=7.5)
expected = uncond + 7.5 * (cond - uncond)
assert torch.allclose(out_s75, expected), "Should follow CFG formula"

print("Warmup 2 passed.")

### Warmup 3: Residual Block (5 min)

Implement `ResidualBlock(nn.Module)` with the following structure:

```
x → Conv2d(C, C, 3, padding=1) → GroupNorm(8, C) → SiLU → Conv2d(C, C, 3, padding=1) → GroupNorm(8, C) → SiLU → + x → output
```

**Interface:**
- `__init__(self, channels: int)`
- `forward(self, x: torch.Tensor) -> torch.Tensor` where x is `(B, C, H, W)`

The skip connection adds the input directly to the output of the second SiLU.

In [ ]:
# Warmup 3: Your implementation



In [ ]:
# Warmup 3: Tests
torch.manual_seed(42)
block = ResidualBlock(channels=32)
x = torch.randn(2, 32, 16, 16)
out = block(x)

assert out.shape == x.shape, f"Expected shape {x.shape}, got {out.shape}"

# Verify spatial dims preserved for different sizes
x2 = torch.randn(1, 32, 64, 64)
out2 = block(x2)
assert out2.shape == x2.shape, f"Spatial dims not preserved: {out2.shape}"

# Verify backward pass
loss = out.sum()
loss.backward()
assert all(p.grad is not None for p in block.parameters()), "All params should have gradients"

print("Warmup 3 passed.")

---

## Main Section (~90 minutes total)

Substantial implementation problems. 30 minutes each. Write clean, tested code.

### Main Problem 1: DiT Block (30 min)

Implement a **Diffusion Transformer (DiT) block** with adaptive layer normalization conditioning.

**Interface:**
```python
class DiTBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int, mlp_ratio: float = 4.0, cond_dim: int = 256) -> None: ...
    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor: ...
```

**Shapes:**
- `x`: `(B, T, D)` — input token sequence
- `cond`: `(B, cond_dim)` — conditioning vector (e.g., timestep + class embedding)
- Output: `(B, T, D)`

**Architecture:**
1. **Conditioning projection:** A single `nn.Linear(cond_dim, 4 * dim)` that produces `(scale1, shift1, scale2, shift2)` — chunk the output into 4 vectors of size `dim`.
2. **Block 1:** Adaptive LN (using scale1, shift1) → Multi-head self-attention → Add residual
3. **Block 2:** Adaptive LN (using scale2, shift2) → MLP (Linear → GELU → Linear, hidden dim = `dim * mlp_ratio`) → Add residual

**Notes:**
- Use `nn.MultiheadAttention(dim, n_heads, batch_first=True)` for the attention — this is an exam, not a from-scratch drill.
- Adaptive LN: apply `nn.LayerNorm(dim)` then modulate with `scale * normalized + shift`. Scale and shift are `(B, dim)`, need unsqueeze to broadcast over T.
- Initialize scale projections to 1 and shift projections to 0 for stable init (or just let the Linear's random init stand — either is acceptable).

In [ ]:
# Main Problem 1: Your implementation



In [ ]:
# Main Problem 1: Tests
torch.manual_seed(42)
B, T, D = 4, 16, 128
n_heads = 8
cond_dim = 256

block = DiTBlock(dim=D, n_heads=n_heads, mlp_ratio=4.0, cond_dim=cond_dim)
x = torch.randn(B, T, D)
cond = torch.randn(B, cond_dim)

out = block(x, cond)

# Shape check
assert out.shape == (B, T, D), f"Expected {(B, T, D)}, got {out.shape}"

# Conditioning should change the output
cond2 = torch.randn(B, cond_dim)
out2 = block(x, cond2)
assert not torch.allclose(out, out2, atol=1e-4), "Different conditioning should produce different output"

# Param count sanity check
n_params = sum(p.numel() for p in block.parameters())
print(f"DiTBlock param count: {n_params:,}")
# For dim=128, n_heads=8, mlp_ratio=4, cond_dim=256:
# Rough estimate: attn ~4*D^2 + cond_proj ~4*D*cond_dim + MLP ~2*(D*4D) + norms
# ~= 4*16384 + 4*128*256 + 2*65536 + small = ~330K
assert 100_000 < n_params < 1_000_000, f"Param count {n_params:,} seems off for dim=128"

# Backward pass
loss = out.sum()
loss.backward()

print("Main Problem 1 passed.")

### Main Problem 2: Temporal Attention for Video (30 min)

Implement a **temporal transformer block** that applies self-attention across the time dimension of a video tensor.

**Interface:**
```python
class TemporalTransformerBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int) -> None: ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

**Shapes:**
- `x`: `(B, C, T, H, W)` — video tensor (batch, channels, frames, height, width)
- Output: `(B, C, T, H, W)` — same shape

**Architecture:**
1. Rearrange input from `(B, C, T, H, W)` → `(B*H*W, T, C)` so that each spatial position has a sequence of T tokens
2. Apply LayerNorm → multi-head self-attention across T
3. Add residual connection (to the pre-norm input)
4. Rearrange back to `(B, C, T, H, W)`

**Notes:**
- Use `nn.MultiheadAttention(dim, n_heads, batch_first=True)` for attention
- The rearrangement is the key part. Think carefully about reshape/permute operations. No einops required — just `permute`, `reshape`, `contiguous`.
- `dim` corresponds to `C` (channel dimension)

In [ ]:
# Main Problem 2: Your implementation



In [ ]:
# Main Problem 2: Tests
torch.manual_seed(42)
B, C, T, H, W = 2, 64, 8, 4, 4

block = TemporalTransformerBlock(dim=C, n_heads=4)
x = torch.randn(B, C, T, H, W)

out = block(x)

# Shape preservation
assert out.shape == (B, C, T, H, W), f"Expected {(B, C, T, H, W)}, got {out.shape}"

# T=1 should give identity-like output (attention over single token is just value projection + residual)
x_single = torch.randn(2, C, 1, 4, 4)
out_single = block(x_single)
assert out_single.shape == x_single.shape, "T=1 shape should be preserved"
# With residual, the output should be close to input (attention on 1 token is near-identity + small projection offset)
diff_single = (out_single - x_single).abs().mean().item()
print(f"T=1 mean abs diff from input: {diff_single:.4f} (should be small-ish due to residual)")

# Attention should mix information across T
x_test = torch.randn(1, C, 4, 2, 2)
# Zero out all but first frame
x_test[:, :, 1:, :, :] = 0.0
out_test = block(x_test)
# After temporal attention, later frames should have non-zero content (info leaked from frame 0)
frame1_energy = out_test[:, :, 1, :, :].abs().mean().item()
assert frame1_energy > 1e-6, f"Temporal attention should propagate info across frames, got energy={frame1_energy}"

# Backward pass
loss = out.sum()
loss.backward()

print("Main Problem 2 passed.")

### Main Problem 3: DDPM Sampling (30 min) — Review Problem

Implement the full **DDPM reverse sampling process** from pure noise to generated images.

**Interface:**
```python
def sample_ddpm(
    model: nn.Module,
    betas: torch.Tensor,
    n_samples: int,
    img_channels: int,
    img_size: int,
    device: torch.device,
) -> torch.Tensor:
    """Generate samples via DDPM reverse process.
    
    Args:
        model: Noise prediction network. Accepts (x_t, t) where t is (B,) integer tensor.
               Returns predicted noise of same shape as x_t.
        betas: 1D tensor of shape (T,) — the noise schedule.
        n_samples: Number of images to generate.
        img_channels: Number of image channels (e.g., 3 for RGB).
        img_size: Spatial size (square images: img_size x img_size).
        device: torch device.
    
    Returns:
        Generated images of shape (n_samples, img_channels, img_size, img_size).
    """
```

**Algorithm:**
1. Precompute from betas: `alphas = 1 - betas`, `alpha_cumprod = cumprod(alphas)`
2. Start with `x_T ~ N(0, I)` of shape `(n_samples, C, H, W)`
3. For `t = T-1, T-2, ..., 0`:
   - Predict noise: `eps = model(x_t, t_tensor)` where `t_tensor` is `(n_samples,)` filled with `t`
   - Compute mean: `mu = (1/sqrt(alpha_t)) * (x_t - (beta_t / sqrt(1 - alpha_bar_t)) * eps)`
   - If `t > 0`: `x_{t-1} = mu + sqrt(beta_t) * z` where `z ~ N(0, I)`
   - If `t == 0`: `x_0 = mu` (no noise added)
4. Return `x_0`

**Notes:**
- Use `torch.no_grad()` context for the entire sampling loop
- The model should be in eval mode

In [ ]:
# Main Problem 3: Your implementation



In [ ]:
# Main Problem 3: Tests
torch.manual_seed(42)

# Simple model that at least transforms input (not identity)
class SimpleNoisePredictor(nn.Module):
    def __init__(self, channels: int = 3) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, channels, 3, padding=1),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return self.net(x)


device = torch.device("cpu")
model = SimpleNoisePredictor(channels=3).to(device)
T = 50  # short schedule for testing
betas = torch.linspace(1e-4, 0.02, T)

# Record the initial noise for comparison
torch.manual_seed(0)
initial_noise = torch.randn(4, 3, 16, 16)

# Generate samples
torch.manual_seed(0)  # same seed so x_T matches initial_noise
samples = sample_ddpm(model, betas, n_samples=4, img_channels=3, img_size=16, device=device)

# Shape check
assert samples.shape == (4, 3, 16, 16), f"Expected (4, 3, 16, 16), got {samples.shape}"

# Output should differ from initial noise (model transforms through reverse process)
assert not torch.allclose(samples, initial_noise, atol=1e-2), "Output should differ from initial noise"

# Output should be finite
assert torch.isfinite(samples).all(), "Output contains NaN or Inf"

print(f"Sample shape: {samples.shape}")
print(f"Sample range: [{samples.min().item():.3f}, {samples.max().item():.3f}]")
print("Main Problem 3 passed.")